In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

: 

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError


def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    
    return response.json()

print(get_json(BASE_URL + 'status.json')['service'])
# 'Northern Star Shop API'

print(get_json(BASE_URL + 'orders/page-99.json'))
# HTTPError: 404 Client Error: Not Found for url: https://cvijan.github.io/nss-api/v2/orders/page-99.json


In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа


def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        if page['next']:
            url = urljoin(url, page['next'])
        else:
            url = None
    
    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(orders) # 1852


In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path:str) -> None:
    with open(path, 'w', encoding= 'utf-8') as f:
        json.dump(records, f, ensure_ascii= False)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))